# Coleta do YouTube: caso Jensen Huang × NVIDIA
Período: 18/10/2026 a 18/11/2026, datas no fuso de Nova York. Vídeos em inglês (por metadado ou detecção de idioma). Uma linha por vídeo e tabelas agregadas por dia e por pregão.

In [ ]:
import os, re, json, time, hashlib, pathlib, unicodedata
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
CHAVES = [k.strip() for k in os.getenv("YOUTUBE_API_KEYS", "").split(",") if k.strip()]
if not CHAVES:
    raise RuntimeError("Defina YOUTUBE_API_KEYS no arquivo .env")

INICIO, FIM = "2026-10-18", "2026-11-18"   # dias no fuso de Nova York, ambos inclusos
FUSO = "America/New_York"
CONSULTA = '"Jensen Huang"'                # frase exata ("|" seria OR e "-" seria NOT na busca do YouTube)
MAX_PAGINAS = 3                            # 50 vídeos por página; 100 unidades de cota por página
URL = "https://www.googleapis.com/youtube/v3"
FMT_UTC = "%Y-%m-%dT%H:%M:%SZ"
COTA_DIARIA_POR_CHAVE = 10_000
FECHAMENTO_BOLSA = 16                      # hora de NY em que a NYSE fecha
FERIADOS_BOLSA = []                        # feriados da NYSE (AAAA-MM-DD); nenhum entre 18/10 e 19/11/2026

RAW, PROC = pathlib.Path("../data/raw"), pathlib.Path("../data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)


def salvar_json(caminho, obj):
    """Escrita atômica: se der erro no meio, o arquivo anterior continua intacto."""
    tmp = caminho.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj), encoding="utf-8")
    tmp.replace(caminho)


def carregar_json(caminho, padrao):
    if not caminho.exists():
        return padrao
    try:
        return json.loads(caminho.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        caminho.replace(caminho.with_suffix(".corrompido"))
        print(f"{caminho.name} estava corrompido; guardado como .corrompido e recomeçando vazio")
        return padrao


# O nome do cache inclui um hash dos parâmetros da busca: mudou algo, o cache é outro.
PARAMS_BUSCA = {"q": CONSULTA, "type": "video", "part": "id", "maxResults": 50, "relevanceLanguage": "en"}
hash_busca = hashlib.md5(json.dumps([PARAMS_BUSCA, MAX_PAGINAS, FUSO, "janela_1d"], sort_keys=True)
                         .encode()).hexdigest()[:8]
cache_arq = RAW / f"cache_ids_{hash_busca}.json"
cache = carregar_json(cache_arq, {})

# Chave ativa lembrada entre execuções (a cota zera à meia-noite do horário do Pacífico).
estado_arq = RAW / "chave_ativa.json"
hoje_pacifico = pd.Timestamp.now(tz="America/Los_Angeles").date().isoformat()
estado = carregar_json(estado_arq, {})
indice = estado["indice"] if estado.get("dia_cota") == hoje_pacifico and estado.get("indice", 0) < len(CHAVES) else 0
print(len(CHAVES), "chave(s) carregada(s), começando pela chave", indice + 1)

## Funções da API
`chamar` tenta de novo em erros de rede, de servidor e de limite de taxa, e troca para a próxima chave quando a cota da atual acaba (a chave ativa é lembrada até a cota zerar). `buscar_ids` faz a busca (100 unidades de cota por página) pedindo vídeos em inglês.

In [ ]:
MOTIVOS_COTA = {"quotaExceeded", "dailyLimitExceeded"}
MOTIVOS_TEMPORARIOS = {"backendError", "internalError", "rateLimitExceeded", "userRateLimitExceeded"}


def trocar_chave():
    """Passa para a próxima chave e a lembra. Devolve False se não há mais chaves."""
    global indice
    if indice >= len(CHAVES) - 1:
        return False
    indice += 1
    salvar_json(estado_arq, {"dia_cota": hoje_pacifico, "indice": indice})
    print(f"Cota esgotada, trocando para a chave {indice + 1}")
    return True


def chamar(endpoint, params, tentativas=5):
    falhas, espera = 0, 2
    while True:
        try:
            resp = requests.get(f"{URL}/{endpoint}", params={**params, "key": CHAVES[indice]}, timeout=30)
            r = resp.json()
        except (requests.RequestException, ValueError) as e:   # rede, timeout ou resposta que não é JSON
            falhas += 1
            if falhas >= tentativas:
                raise RuntimeError(f"{endpoint}: falha após {tentativas} tentativas ({e})") from e
            time.sleep(espera)
            espera *= 2
            continue
        if "error" not in r:
            return r
        erro = r["error"]
        motivo = (erro.get("errors") or [{}])[0].get("reason", "desconhecido")
        if motivo in MOTIVOS_COTA:
            if trocar_chave():
                continue
            raise RuntimeError("Cota esgotada em todas as chaves. O cache guardou o progresso: "
                               "rode de novo depois que a cota zerar (meia-noite do Pacífico).")
        if (motivo in MOTIVOS_TEMPORARIOS or resp.status_code >= 500) and falhas + 1 < tentativas:
            falhas += 1
            time.sleep(espera)
            espera *= 2
            continue
        raise RuntimeError(f"{motivo}: {erro.get('message', '')}")


def buscar_ids(ini, fim):
    ids, token = [], None
    for _ in range(MAX_PAGINAS):
        params = {**PARAMS_BUSCA, "publishedAfter": ini, "publishedBefore": fim}
        if token:
            params["pageToken"] = token
        r = chamar("search", params)
        ids += [i["id"]["videoId"] for i in r.get("items", []) if i.get("id", {}).get("videoId")]
        token = r.get("nextPageToken")
        if not token:
            break
    return ids

## Busca dos IDs (gasta cota)
Uma janela por dia de Nova York (meia-noite a meia-noite, convertida para UTC), então não há margem e nenhuma janela gasta o teto de vídeos com dias vizinhos. O cache guarda cada janela já buscada, e o nome do arquivo inclui um hash dos parâmetros da busca (consulta, páginas, idioma, fuso): mudar qualquer um deles cria um cache novo em vez de reaproveitar IDs antigos. Uma execução completa custa até 32 × `MAX_PAGINAS` × 100 unidades de cota. Apague `data/raw/cache_ids_*.json` (na raiz do projeto) só se quiser refazer a busca do zero; o antigo `cache_ids.json`, de janelas de 3 dias, não é mais usado.

In [ ]:
# Uma janela por dia de NY: da meia-noite local (convertida para UTC) até a meia-noite seguinte.
janelas = []
for dia in pd.date_range(INICIO, FIM):
    ini_ny = pd.Timestamp(dia.date(), tz=FUSO)
    fim_ny = pd.Timestamp((dia + pd.Timedelta(days=1)).date(), tz=FUSO)
    janelas.append((ini_ny.tz_convert("UTC").strftime(FMT_UTC), fim_ny.tz_convert("UTC").strftime(FMT_UTC)))

pendentes = [j for j in janelas if j[0] not in cache]
custo = len(pendentes) * MAX_PAGINAS * 100
disponivel = len(CHAVES) * COTA_DIARIA_POR_CHAVE
print(f"{len(janelas)} janelas, {len(pendentes)} fora do cache: até {custo} unidades de cota (disponível: {disponivel})")
if custo > disponivel:
    print("Aviso: não cabe em um dia. A busca para quando a cota acabar e continua de onde parou na próxima execução.")

ids = set()
for ini, fim in janelas:
    if ini not in cache:
        cache[ini] = buscar_ids(ini, fim)
        salvar_json(cache_arq, cache)
        time.sleep(0.2)
    ids.update(cache[ini])
ids = sorted(ids)
print(len(ids), "IDs encontrados")

## Detalhes dos vídeos
Título, canal, data de publicação, descrição (primeiros 1.000 caracteres), tags, idioma declarado, views, curtidas e comentários. Custa 1 unidade de cota a cada 50 vídeos.

In [ ]:
linhas = []
for i in range(0, len(ids), 50):
    r = chamar("videos", {"part": "snippet,statistics", "id": ",".join(ids[i:i + 50])})
    for v in r.get("items", []):
        s, sn = v.get("statistics", {}), v["snippet"]
        linhas.append({
            "video_id": v["id"],
            "data_publicacao": sn["publishedAt"],
            "titulo": sn.get("title", ""),
            "canal": sn.get("channelTitle", ""),
            "descricao": sn.get("description", "")[:1000],
            "tags": "|".join(sn.get("tags", [])),
            "idioma_audio": sn.get("defaultAudioLanguage"),
            "idioma_padrao": sn.get("defaultLanguage"),
            "views": s.get("viewCount"),
            "curtidas": s.get("likeCount"),
            "comentarios": s.get("commentCount"),
        })
if not linhas:
    raise RuntimeError("A API não devolveu nenhum vídeo: confira a consulta, o período e as chaves.")
bruto = pd.DataFrame(linhas)
print(len(bruto), "vídeos brutos")

## Limpeza, filtros e conversão de fuso
1. Remove duplicados e converte os números (curtidas ocultas e comentários desativados ficam NaN e valem 0 só no cálculo da taxa).
2. **Conversão de fuso:** `hora_utc` é a hora original da API, `hora_nova_york` é a mesma hora no fuso da bolsa, e `data` é o dia de `hora_nova_york`.
3. Corta o período de 18/10 a 18/11 usando essa `data`.
4. **Tema:** mantém o vídeo se "Jensen Huang" aparece no título, se "NVIDIA" aparece no título, ou se "Jensen Huang" aparece na descrição ou nas tags. A coluna `tema_origem` diz qual regra valeu, para auditar.
5. **Idioma:** usa `defaultAudioLanguage` ou `defaultLanguage`; se ambos estiverem vazios, detecta o idioma do título e da descrição (precisa de `pip install langdetect`). Mantém inglês e vídeos de idioma indefinido (`idioma_fonte = indefinido`); descarta o resto.
6. Calcula a taxa de engajamento. O dicionário `funil` guarda quantos vídeos saíram em cada etapa. O CSV é salvo na célula seguinte.

In [ ]:
try:
    from langdetect import detect, DetectorFactory, LangDetectException
    DetectorFactory.seed = 0
    TEM_DETECTOR = True
except ImportError:
    TEM_DETECTOR = False
    print("langdetect não instalado (pip install langdetect): vídeos sem metadado de idioma ficam "
          "como 'indefinido' e são mantidos.")

PESSOA = re.compile(r"jens[eo]n\s+huang", re.I)
EMPRESA = re.compile(r"nvidia", re.I)


def como_texto(x):
    return x if isinstance(x, str) else ""


def exigir_dados(d, etapa):
    if d.empty:
        raise RuntimeError(f"Nenhum vídeo sobrou depois da etapa: {etapa}")


def origem_tema(r):
    if PESSOA.search(r["titulo"]):
        return "titulo_pessoa"
    if EMPRESA.search(r["titulo"]):
        return "titulo_empresa"
    if PESSOA.search(f"{r['descricao']} {r['tags']}"):
        return "descricao_tags"
    return None


def idioma_do_video(r):
    for c in ("idioma_audio", "idioma_padrao"):
        v = como_texto(r[c]).strip().lower()
        if v and v not in ("und", "zxx"):          # "und" e "zxx" = indeterminado / sem fala
            return v.split("-")[0], "metadata"
    if TEM_DETECTOR:
        t = re.sub(r"https?://\S+", " ", f"{r['titulo']}. {r['descricao'][:200]}")
        try:
            return detect(t), "detectado"
        except LangDetectException:
            pass
    return None, "indefinido"


funil = {"ids_encontrados": len(ids), "videos_brutos": len(bruto)}
df = bruto.drop_duplicates("video_id").copy()
funil["duplicados"] = len(bruto) - len(df)

for c in ["views", "curtidas", "comentarios"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["hora_utc"] = pd.to_datetime(df["data_publicacao"], utc=True)
df["hora_nova_york"] = df["hora_utc"].dt.tz_convert(FUSO)
df["data"] = df["hora_nova_york"].dt.date

no_periodo = (df["data"] >= pd.to_datetime(INICIO).date()) & (df["data"] <= pd.to_datetime(FIM).date())
funil["fora_do_periodo"] = int((~no_periodo).sum())
df = df[no_periodo].copy()
exigir_dados(df, "corte do período")

df["tema_origem"] = df.apply(origem_tema, axis=1)
funil["fora_do_tema"] = int(df["tema_origem"].isna().sum())
df = df[df["tema_origem"].notna()].copy()
exigir_dados(df, "filtro de tema")

idiomas = df.apply(idioma_do_video, axis=1, result_type="expand")
df["idioma"], df["idioma_fonte"] = idiomas[0], idiomas[1]
fora_idioma = df["idioma"].notna() & (df["idioma"] != "en")
funil["fora_do_idioma"] = int(fora_idioma.sum())
funil["idioma_indefinido_mantidos"] = int(df["idioma"].isna().sum())
df = df[~fora_idioma].copy()
exigir_dados(df, "filtro de idioma")

df["taxa_engajamento"] = (df["curtidas"].fillna(0) + df["comentarios"].fillna(0)) \
    / df["views"].where(df["views"] > 0)
df["data_coleta"] = pd.Timestamp.now(tz=FUSO).date()

df = df.sort_values("hora_utc").reset_index(drop=True)
funil["videos_finais"] = len(df)
print(pd.Series(funil).to_string())

## Padronização de texto, features de tempo e outliers
- `dia_semana` e `hora_publicacao` vêm do horário de Nova York.
- `data_pregao` é o pregão da NYSE que "recebe" o vídeo: o próprio dia, se for dia útil e a publicação for antes das 16h; caso contrário, o próximo dia útil. Assim, vídeos de fim de semana e do fim da tarde entram no cruzamento com a bolsa em vez de serem descartados. Feriados da bolsa vão em `FERIADOS_BOLSA` (não há nenhum entre 18/10 e 19/11/2026).
- `titulo_limpo` fica em minúsculas, sem acentos e sem emojis ou pontuação. O `titulo` original é mantido.
- `outlier_views` marca vídeos com views muito acima dos demais, pela regra do intervalo interquartil aplicada ao log10 das views. **Decisão:** os outliers são marcados e mantidos, e a análise compara os resultados com e sem eles, porque um vídeo viral pode distorcer as somas e médias do dia.
- Esta célula salva `data/raw/youtube_videos.csv`.

In [ ]:
bdia = pd.offsets.CustomBusinessDay(holidays=FERIADOS_BOLSA)


def data_pregao(ts):
    """Pregão que recebe a publicação: o mesmo dia (útil, antes do fechamento) ou o próximo dia útil."""
    d = pd.Timestamp(ts.date())
    if ts.hour >= FECHAMENTO_BOLSA or not bdia.is_on_offset(d):
        return (d + bdia).date()
    return d.date()


df["dia_semana"] = df["hora_nova_york"].dt.day_name()
df["hora_publicacao"] = df["hora_nova_york"].dt.hour
df["data_pregao"] = df["hora_nova_york"].apply(data_pregao)


def limpar_titulo(t):
    t = unicodedata.normalize("NFKD", str(t))
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^\w\s]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()


df["titulo_limpo"] = df["titulo"].apply(limpar_titulo)

log_views = np.log10(df["views"] + 1)
q1, q3 = log_views.quantile([0.25, 0.75])
df["outlier_views"] = log_views > q3 + 1.5 * (q3 - q1)
print(df["outlier_views"].sum(), "vídeos marcados como outlier de views")
print((df["data_pregao"] != df["data"]).sum(), "vídeos deslocados para o pregão seguinte (fim de semana ou após o fechamento)")

df.to_csv(RAW / "youtube_videos.csv", index=False)

## Agregação por dia e por pregão
A taxa é ponderada: soma(curtidas + comentários) / soma(views). Dias sem vídeo aparecem com zeros e taxa vazia, em vez de sumirem.
- `youtube_diario.csv`: agregado por `data` (dia de Nova York). `dia_util` marca segunda a sexta.
- `youtube_diario_pregao.csv`: agregado por `data_pregao`, para o join com a bolsa. `cobertura_completa` é falso nas pontas: o primeiro pregão (19/10) não recebe a sexta depois do fechamento nem o sábado, porque a coleta começa em 18/10, e o pregão de 19/11 só recebe o que foi publicado depois do fechamento de 18/11. Exclua ou sinalize esses dias na análise.

In [ ]:
def agregar(base, coluna, calendario):
    g = base.groupby(coluna).agg(
        n_videos=("video_id", "count"), views=("views", "sum"),
        curtidas=("curtidas", "sum"), comentarios=("comentarios", "sum"))
    g.index = pd.to_datetime(g.index)
    g = g.reindex(calendario, fill_value=0)          # dia sem vídeo vira linha de zeros
    g["taxa_engajamento"] = (g["curtidas"] + g["comentarios"]) / g["views"].where(g["views"] > 0)
    g.index.name = coluna
    return g.reset_index()


# Por dia de calendário (NY)
diario = agregar(df, "data", pd.date_range(INICIO, FIM))
diario["dia_util"] = diario["data"].dt.dayofweek < 5
diario["data"] = diario["data"].dt.date
diario.to_csv(PROC / "youtube_diario.csv", index=False)

# Por pregão da NYSE
ultimo = max(pd.Timestamp(FIM), pd.Timestamp(df["data_pregao"].max()))
pregoes = pd.bdate_range(INICIO, ultimo, freq="C", holidays=FERIADOS_BOLSA)
diario_pregao = agregar(df, "data_pregao", pregoes)
anterior = pd.Series([d - bdia for d in diario_pregao["data_pregao"]])
diario_pregao["cobertura_completa"] = ((anterior >= pd.Timestamp(INICIO)).values
                                       & (diario_pregao["data_pregao"] <= pd.Timestamp(FIM)).values)
diario_pregao["data_pregao"] = diario_pregao["data_pregao"].dt.date
diario_pregao.to_csv(PROC / "youtube_diario_pregao.csv", index=False)

print(len(diario), "dias |", int(diario["dia_util"].sum()), "dias úteis |",
      len(diario_pregao), "pregões,", int(diario_pregao["cobertura_completa"].sum()), "com cobertura completa")

## Conferência
Confira o menor e o maior dia, os valores ausentes e uma amostra de títulos (para ver se há ruído fora do tema ou fora do inglês).

In [ ]:
print(df["data"].min(), "->", df["data"].max())
print(df[["views", "curtidas", "comentarios"]].isna().sum())
df[["hora_utc", "hora_nova_york", "data", "data_pregao", "titulo", "canal", "idioma", "tema_origem",
    "views", "curtidas", "comentarios", "taxa_engajamento"]].head(10)

In [ ]:
diario

In [ ]:
diario_pregao

In [ ]:
for ini, _ in janelas:
    n = len(cache.get(ini, []))
    print(ini[:10], n, "<- teto atingido" if n >= 50 * MAX_PAGINAS else "")

## Validação dos CSVs salvos
Lê os arquivos que serão entregues e confere duplicados, valores impossíveis, somas entre as tabelas, o fuso e a coerência de `data` e `data_pregao`. Os links de três vídeos sorteados servem para conferir título, canal e views à mão no YouTube.

In [ ]:
salvo = pd.read_csv(RAW / "youtube_videos.csv")
diario_salvo = pd.read_csv(PROC / "youtube_diario.csv")
pregao_salvo = pd.read_csv(PROC / "youtube_diario_pregao.csv")

print("vídeos:", len(salvo), "| dias:", len(diario_salvo), "| pregões:", len(pregao_salvo))
print("video_id duplicados:", salvo["video_id"].duplicated().sum())
print("views negativas:", (salvo["views"] < 0).sum())
print("curtidas maiores que views:", (salvo["curtidas"] > salvo["views"]).sum())
print("taxa fora de 0 a 1:", (~salvo["taxa_engajamento"].dropna().between(0, 1)).sum())
for nome, t in [("diário", diario_salvo), ("pregão", pregao_salvo)]:
    print(f"tabela {nome}: views iguais às dos vídeos:", round(salvo["views"].sum()) == round(t["views"].sum()),
          "| soma de n_videos == nº de vídeos:", len(salvo) == t["n_videos"].sum())

salvo["fuso"] = salvo["hora_nova_york"].str[-6:]
print(salvo.groupby("fuso")["data"].agg(["min", "max"]))      # -04:00 até 31/10 e -05:00 a partir de 01/11
ny = pd.to_datetime(salvo["hora_utc"], utc=True).dt.tz_convert(FUSO)
print("data confere com hora_utc convertida:", (ny.dt.strftime("%Y-%m-%d") == salvo["data"]).all())
print("data_pregao sempre em dia útil:", (pd.to_datetime(salvo["data_pregao"]).dt.dayofweek < 5).all())
print("data_pregao nunca antes de data:", (salvo["data_pregao"] >= salvo["data"]).all())

print(salvo["idioma"].value_counts(dropna=False).head(8))
print(salvo["tema_origem"].value_counts())
print(salvo.nlargest(5, "views")[["data", "titulo", "canal", "views"]])

for _, r in salvo.sample(min(3, len(salvo)), random_state=1).iterrows():
    print(r["titulo"][:60], "|", r["canal"], "|", r["data"], "|", r["views"], "|",
          "https://www.youtube.com/watch?v=" + r["video_id"])

## Estatísticas para o relatório
Os números da documentação abaixo saem desta célula. Rode-a antes de citá-los.

In [ ]:
teto = 50 * MAX_PAGINAS
no_teto = sum(len(cache.get(ini, [])) >= teto for ini, _ in janelas)
sem_curtidas, sem_comentarios = int(df["curtidas"].isna().sum()), int(df["comentarios"].isna().sum())

print(f"Janelas diárias: {len(janelas)} | que atingiram o teto de {teto} vídeos: {no_teto}")
print("Funil:", funil)
print(f"Sem curtidas: {sem_curtidas} ({sem_curtidas / len(df):.1%}) | "
      f"sem comentários: {sem_comentarios} ({sem_comentarios / len(df):.1%})")
print("Origem do idioma:", df["idioma_fonte"].value_counts().to_dict())
print("Origem do tema:", df["tema_origem"].value_counts().to_dict())
print("Deslocados para o pregão seguinte:", int((df["data_pregao"] != df["data"]).sum()),
      "| pregões com cobertura completa:", int(diario_pregao["cobertura_completa"].sum()), "de", len(diario_pregao))

## Documentação da fonte e limitações (para o relatório)
**Fonte:** YouTube Data API v3 (https://developers.google.com/youtube/v3), acessada na data da coluna `data_coleta`. Período: 18/10 a 18/11/2026, datas no fuso de Nova York.

**Por que junto com as outras bases:** a repercussão no YouTube, a cobertura de notícias e as ações da NVDA têm a data como chave comum, o que permite comparar atenção do público, da imprensa e do mercado dia a dia.

**Limitações** (os números vêm da célula "Estatísticas para o relatório"):
1. A busca é limitada a `MAX_PAGINAS` páginas de 50 vídeos por dia (teto de 150 com 3 páginas; é uma escolha do notebook para controlar a cota, a API permitiria mais). Nos dias em que o teto é atingido, o dataset é uma amostra dos vídeos mais relevantes e `n_videos` não representa o volume real de publicações. Como a busca ordena por relevância, a amostra tende a favorecer vídeos de maior audiência. A análise usa `views` e `taxa_engajamento`.
2. O filtro de tema seleciona vídeos sobre Jensen Huang e a NVIDIA em geral, não só sobre o caso: títulos com "NVIDIA" incluem reviews de GPU e análises de ações, e vídeos sem o nome no título só entram se a descrição ou as tags citarem "Jensen Huang". Vídeos de grande audiência (como um keynote) podem criar picos de views que não refletem o caso.
3. Curtidas ocultas e comentários desativados ficam como NaN nas colunas originais e valem 0 só no cálculo da taxa de engajamento, o que a subestima levemente.
4. As views são cumulativas até a data da coleta, então vídeos mais antigos tiveram mais tempo para acumular.
5. O idioma vem do que o canal declarou (`defaultAudioLanguage` ou `defaultLanguage`), que pode estar errado ou ausente; nesses casos usa-se detecção automática sobre título e descrição, menos confiável em textos curtos. Vídeos de idioma indefinido são mantidos (`idioma_fonte = indefinido`).
6. Publicações de fim de semana e depois das 16h (NY) são atribuídas ao próximo pregão, então a atenção de um dia de bolsa reúne o que foi publicado desde o fechamento anterior. Os pregões das pontas (`cobertura_completa` falso) têm cobertura parcial. O relatório deve informar quantos vídeos foram deslocados para o pregão seguinte.
